# Experiment 5: Orbit structure and the loss landscape

The $\mathrm{GL}(n)$ symmetry of a deep linear network means the loss function is **constant along orbits** of the symmetry group. If we train a network and arrive at weights $(W_1, W_2, W_3)$, then every reparametrization $(K W_1, W_2 K^{-1}, W_3)$ has exactly the same loss. These orbits form flat valleys in the loss landscape.

This experiment makes the reparametrization problem viscerally clear:
1. Train a deep linear network on a regression task
2. Move along the orbit (apply $K(t) = e^{tA}$ reparametrization) → loss stays flat
3. Move perpendicular to the orbit (change the actual function) → loss changes
4. Visualize the 2D loss landscape: orbit direction vs function-changing direction


In [1]:
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import cm
from scipy.linalg import expm

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'figure.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

torch.manual_seed(42)
np.random.seed(42)

# Simple regression task: learn a linear map y = W_true @ x + noise
d_in, d_hidden, d_out = 4, 6, 3
n_samples = 200

W_true = np.random.randn(d_out, d_in)
X = np.random.randn(n_samples, d_in)
Y = X @ W_true.T + 0.01 * np.random.randn(n_samples, d_out)

print(f"Task: learn a linear map R^{d_in} -> R^{d_out}")
print(f"Training data: {n_samples} samples")
print(f"Network: 3-layer deep linear (dims: {d_in} -> {d_hidden} -> {d_hidden} -> {d_out})")


Task: learn a linear map R^4 -> R^3
Training data: 200 samples
Network: 3-layer deep linear (dims: 4 -> 6 -> 6 -> 3)


## 1. Train a deep linear network

We train a 3-layer deep linear network $f(x) = W_3 W_2 W_1 x$ using gradient descent. The network will converge to some factorization of the true linear map, but this factorization is not unique — the entire $\mathrm{GL}(n_\mathrm{hidden})$ orbit gives the same function.


In [2]:
# Train with numpy (simple gradient descent)
lr = 0.001
n_epochs = 5000

# Initialize weights
W1 = 0.1 * np.random.randn(d_hidden, d_in)
W2 = 0.1 * np.random.randn(d_hidden, d_hidden)
W3 = 0.1 * np.random.randn(d_out, d_hidden)

def compute_loss(W1, W2, W3, X, Y):
    pred = X @ W1.T @ W2.T @ W3.T
    return ((pred - Y) ** 2).mean()

losses = []
for epoch in range(n_epochs):
    # Forward
    h1 = X @ W1.T                    # (n, d_hidden)
    h2 = h1 @ W2.T                   # (n, d_hidden)
    pred = h2 @ W3.T                  # (n, d_out)
    residual = pred - Y               # (n, d_out)
    loss = (residual ** 2).mean()
    losses.append(loss)
    
    # Backward (manual gradients)
    dL_dpred = 2 * residual / n_samples   # (n, d_out)
    dL_dW3 = dL_dpred.T @ h2              # (d_out, d_hidden)
    dL_dh2 = dL_dpred @ W3                # (n, d_hidden)
    dL_dW2 = dL_dh2.T @ h1               # (d_hidden, d_hidden)
    dL_dh1 = dL_dh2 @ W2                  # (n, d_hidden)
    dL_dW1 = dL_dh1.T @ X                # (d_hidden, d_in)
    
    W1 -= lr * dL_dW1
    W2 -= lr * dL_dW2
    W3 -= lr * dL_dW3

final_loss = compute_loss(W1, W2, W3, X, Y)
W_total = W3 @ W2 @ W1
print(f"Final loss: {final_loss:.6f}")
print(f"||W_total - W_true||_F = {np.linalg.norm(W_total - W_true):.6f}")


Final loss: 0.000243
||W_total - W_true||_F = 0.023350


## 2. Moving along the orbit: loss stays flat

We parameterize movement along the orbit using the matrix exponential. For a random matrix $A \in \mathfrak{gl}(n)$, the curve $K(t) = e^{tA}$ traces a path through $\mathrm{GL}(n)$. The reparametrized weights $(K(t) W_1, W_2 K(t)^{-1}, W_3)$ all compute the same function, so the loss is constant.

We compare this with movement that *changes* the function. Given a random unit-norm direction $\Delta \in \mathbb{R}^{d_{\mathrm{out}} \times d_{\mathrm{in}}}$, we modify $W_3$ so that the total map shifts: $W_{\mathrm{total}}(s) = W_3 W_2 W_1 + s \cdot \Delta$. Concretely, $W_3(s) = W_3 + s \cdot \Delta \cdot (W_2 W_1)^+$ where $(W_2 W_1)^+$ is the pseudoinverse, ensuring the product changes by exactly $s \cdot \Delta$.

**Reading the figure.** The left panel plots loss vs the orbit parameter $t \in [-2, 2]$. All values of $t$ lie on *the same orbit* — they are different weight configurations $(K(t)W_1, W_2 K(t)^{-1}, W_3)$ that compute the same function $W_3 W_2 W_1$. The curve is flat to machine precision ($\sim 10^{-16}$ standard deviation). Any small oscillation visible at the tails ($|t| \to 2$) is numerical: $e^{tA}$ has large entries there, and floating-point cancellation in $K(t) K(t)^{-1}$ introduces rounding errors.

The right panel plots loss vs the function perturbation parameter $s \in [-0.5, 0.5]$. Each value of $s$ gives a *different* orbit (a different function). At $s = 0$ the network computes the trained solution, which is near-optimal, so the loss is at its minimum (not zero — it equals the training loss $\approx 2.4 \times 10^{-4}$). Moving in either direction ($s > 0$ or $s < 0$) shifts the linear map away from the optimum, increasing the loss quadratically. The parabolic shape reflects the quadratic nature of the MSE loss near a minimum.


In [3]:
# Generate a random direction in gl(n) (the Lie algebra)
A_orbit = np.random.randn(d_hidden, d_hidden)

# Also generate a "perpendicular" direction that changes the actual function
# We perturb W_total directly: W_total + s * Delta
Delta = np.random.randn(d_out, d_in)
Delta = Delta / np.linalg.norm(Delta)  # normalize

t_range = np.linspace(-2, 2, 100)
s_range = np.linspace(-0.5, 0.5, 100)

# Loss along orbit direction (reparametrization, should be flat)
orbit_losses = []
for t in t_range:
    Kt = expm(t * A_orbit)
    Kt_inv = expm(-t * A_orbit)
    W1_t = Kt @ W1
    W2_t = W2 @ Kt_inv
    orbit_losses.append(compute_loss(W1_t, W2_t, W3, X, Y))

# Loss along function-changing direction (perturb W3 to change W_total)
# We modify W3 so that W_total changes: W3_new = W3 + s * Delta @ (W2 @ W1)^{-1}
# This way W_total_new = (W3 + s * Delta @ (W2@W1)^-1) @ W2 @ W1 = W_total + s * Delta
W21 = W2 @ W1
W21_pinv = np.linalg.pinv(W21)

func_losses = []
for s in s_range:
    W3_s = W3 + s * (Delta @ W21_pinv)
    func_losses.append(compute_loss(W1, W2, W3_s, X, Y))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(t_range, orbit_losses, color='#2E86C1', linewidth=2)
ax.set_xlabel('$t$ (orbit parameter)')
ax.set_ylabel('Loss')
ax.set_title('Along the orbit: $K(t) = e^{tA}$ reparametrization\n(loss is constant — flat valley)')
ax.grid(True, alpha=0.3)
# Show how flat it is
orbit_std = np.std(orbit_losses)
ax.text(0.02, 0.98, f'Loss std: {orbit_std:.2e}', transform=ax.transAxes,
        fontsize=10, verticalalignment='top', color='#27AE60')

ax = axes[1]
ax.plot(s_range, func_losses, color='#E74C3C', linewidth=2)
ax.set_xlabel('$s$ (function perturbation)')
ax.set_ylabel('Loss')
ax.set_title('Perpendicular to orbit: changing $W_{\\mathrm{total}}$\n(loss varies — steep walls)')
ax.grid(True, alpha=0.3)

plt.suptitle('Loss landscape: orbit direction vs function-changing direction', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('Ex5_fig_orbit_vs_perpendicular.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print("Saved: Ex5_fig_orbit_vs_perpendicular.png")


Saved: Ex5_fig_orbit_vs_perpendicular.png


/var/folders/xh/h88wz4193gzg53_x42zm8ngm0000gn/T/ipykernel_23192/3237216039.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. 2D loss landscape heatmap

We create a 2D slice of the loss landscape with the orbit direction on one axis and the function-changing direction on the other. The result should show a **flat valley** along the orbit direction and **steep walls** perpendicular to it.


In [4]:
# 2D grid: t (orbit) x s (function change)
t_grid = np.linspace(-2, 2, 80)
s_grid = np.linspace(-0.3, 0.3, 80)
loss_2d = np.zeros((len(s_grid), len(t_grid)))

for i, s in enumerate(s_grid):
    W3_s = W3 + s * (Delta @ W21_pinv)
    for j, t in enumerate(t_grid):
        Kt = expm(t * A_orbit)
        Kt_inv = expm(-t * A_orbit)
        W1_t = Kt @ W1
        W2_t = W2 @ Kt_inv
        loss_2d[i, j] = compute_loss(W1_t, W2_t, W3_s, X, Y)

fig, ax = plt.subplots(figsize=(10, 7))

# Use log scale for better visualization
loss_plot = np.log10(loss_2d + 1e-10)
im = ax.imshow(loss_plot, extent=[t_grid[0], t_grid[-1], s_grid[0], s_grid[-1]],
               aspect='auto', origin='lower', cmap='viridis')
ax.set_xlabel('$t$ (orbit direction: reparametrization)', fontsize=13)
ax.set_ylabel('$s$ (perpendicular: function change)', fontsize=13)
ax.set_title('2D loss landscape slice\n(flat valley along orbit, steep perpendicular)', fontsize=14)

cbar = plt.colorbar(im, ax=ax, label='$\\log_{10}$(Loss)')

# Mark the trained solution
ax.plot(0, 0, '*', color='white', markersize=15, markeredgecolor='black', markeredgewidth=1)
ax.annotate('Trained\nsolution', xy=(0, 0), xytext=(0.8, 0.15),
            fontsize=11, color='white', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='white', lw=2))

# Draw orbit direction arrow
ax.annotate('', xy=(1.8, 0), xytext=(-1.8, 0),
            arrowprops=dict(arrowstyle='->', color='#E74C3C', lw=2, linestyle='--'))
ax.text(1.5, -0.05, 'Orbit\n(flat)', fontsize=10, color='#E74C3C', ha='center', va='top')

plt.tight_layout()
plt.savefig('Ex5_fig_2d_loss_landscape.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print("Saved: Ex5_fig_2d_loss_landscape.png")


Saved: Ex5_fig_2d_loss_landscape.png


/var/folders/xh/h88wz4193gzg53_x42zm8ngm0000gn/T/ipykernel_23192/4228988322.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Multiple orbit directions

The orbit is not one-dimensional — it has dimension $\dim(\mathrm{GL}(n)) = n^2$. We verify that the loss is flat along *multiple* independent orbit directions.


In [5]:
n_directions = 10
fig, ax = plt.subplots(figsize=(10, 5))

colors = plt.cm.viridis(np.linspace(0.2, 0.8, n_directions))

for k in range(n_directions):
    A_k = np.random.randn(d_hidden, d_hidden)
    losses_k = []
    for t in t_range:
        Kt = expm(t * A_k)
        Kt_inv = expm(-t * A_k)
        W1_t = Kt @ W1
        W2_t = W2 @ Kt_inv
        losses_k.append(compute_loss(W1_t, W2_t, W3, X, Y))
    ax.plot(t_range, losses_k, color=colors[k], linewidth=1.5, alpha=0.7,
            label=f'Direction {k+1}' if k < 3 else None)

ax.set_xlabel('$t$ (orbit parameter)')
ax.set_ylabel('Loss')
ax.set_title(f'Loss along {n_directions} random orbit directions\n(all flat — the orbit is {d_hidden}²={d_hidden**2}-dimensional)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Report statistics
all_stds = []
for k in range(n_directions):
    A_k = np.random.randn(d_hidden, d_hidden)
    losses_k = [compute_loss(expm(t*A_k) @ W1, W2 @ expm(-t*A_k), W3, X, Y) for t in t_range]
    all_stds.append(np.std(losses_k))
ax.text(0.02, 0.98, f'Mean loss std across directions: {np.mean(all_stds):.2e}',
        transform=ax.transAxes, fontsize=10, verticalalignment='top', color='#27AE60')

plt.tight_layout()
plt.savefig('Ex5_fig_multiple_orbit_directions.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print("Saved: Ex5_fig_multiple_orbit_directions.png")


Saved: Ex5_fig_multiple_orbit_directions.png


/var/folders/xh/h88wz4193gzg53_x42zm8ngm0000gn/T/ipykernel_23192/2536996086.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Training curve with orbit visualization

We show the training trajectory projected onto the orbit and perpendicular directions. During training, the network moves both along orbits (redundant movement) and across orbits (useful learning). The orbit component is wasted computation — the network is exploring equivalent solutions.


In [6]:
# Retrain and record the trajectory of W_total at each epoch
lr = 0.001
n_epochs = 3000

W1_t = 0.1 * np.random.randn(d_hidden, d_in)
W2_t = 0.1 * np.random.randn(d_hidden, d_hidden)
W3_t = 0.1 * np.random.randn(d_out, d_hidden)

W_total_trajectory = []
W2_trajectory = []
loss_trajectory = []

for epoch in range(n_epochs):
    h1 = X @ W1_t.T
    h2 = h1 @ W2_t.T
    pred = h2 @ W3_t.T
    residual = pred - Y
    loss = (residual ** 2).mean()
    loss_trajectory.append(loss)
    
    W_total_trajectory.append((W3_t @ W2_t @ W1_t).copy())
    W2_trajectory.append(W2_t.copy())
    
    dL_dpred = 2 * residual / n_samples
    dL_dW3 = dL_dpred.T @ h2
    dL_dh2 = dL_dpred @ W3_t
    dL_dW2 = dL_dh2.T @ h1
    dL_dh1 = dL_dh2 @ W2_t
    dL_dW1 = dL_dh1.T @ X
    
    W1_t -= lr * dL_dW1
    W2_t -= lr * dL_dW2
    W3_t -= lr * dL_dW3

# Compute how much W_total changes vs how much W2 changes
# W_total changes = useful learning; W2 changes beyond W_total = orbit movement
W_total_diffs = [np.linalg.norm(W_total_trajectory[i+1] - W_total_trajectory[i]) 
                 for i in range(len(W_total_trajectory)-1)]
W2_diffs = [np.linalg.norm(W2_trajectory[i+1] - W2_trajectory[i]) 
            for i in range(len(W2_trajectory)-1)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogy(loss_trajectory, color='#2E86C1', linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Training loss')
ax.grid(True, alpha=0.3)

ax = axes[1]
# Smooth with moving average
window = 50
W_total_smooth = np.convolve(W_total_diffs, np.ones(window)/window, mode='valid')
W2_smooth = np.convolve(W2_diffs, np.ones(window)/window, mode='valid')

ax.plot(W_total_smooth, label='$\\|\\Delta W_{\\mathrm{total}}\\|$ (useful learning)', 
        color='#27AE60', linewidth=1.5)
ax.plot(W2_smooth, label='$\\|\\Delta W_2\\|$ (includes orbit movement)', 
        color='#E74C3C', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('Step size (smoothed)')
ax.set_title('Parameter movement: useful vs redundant')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.savefig('Ex5_fig_training_orbit_movement.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print("Saved: Ex5_fig_training_orbit_movement.png")


Saved: Ex5_fig_training_orbit_movement.png


/var/folders/xh/h88wz4193gzg53_x42zm8ngm0000gn/T/ipykernel_23192/4260593640.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Gradient decomposition: orbit vs cross-orbit contribution to loss

Sections 2--4 showed that the loss is constant along orbits. A stronger test: decompose the gradient itself into orbit and cross-orbit components, then measure how much loss each component is responsible for.

The orbit tangent space at $(W_1, W_2)$ under the $\mathrm{GL}(n)$ action $(W_1, W_2) \mapsto (KW_1, W_2 K^{-1})$ is
$$T_{\mathrm{orbit}} = \{(AW_1,\; -W_2 A) : A \in \mathfrak{gl}(n)\}.$$
To project the joint gradient $(\nabla_{W_1}\mathcal{L},\; \nabla_{W_2}\mathcal{L})$ onto this subspace, we solve for the $A$ that best approximates the gradient in the orbit direction:
$$A^* = \arg\min_A \; \|AW_1 - \nabla_{W_1}\mathcal{L}\|_F^2 + \|{-W_2 A} - \nabla_{W_2}\mathcal{L}\|_F^2.$$
This is a linear least-squares problem in $\mathrm{vec}(A)$, solved via the Kronecker product formulation:
$$\begin{pmatrix} W_1^\top \otimes I \\ -I \otimes W_2 \end{pmatrix} \mathrm{vec}(A) = \begin{pmatrix} \mathrm{vec}(\nabla_{W_1}\mathcal{L}) \\ \mathrm{vec}(\nabla_{W_2}\mathcal{L}) \end{pmatrix}.$$
The orbit component is $(A^* W_1, -W_2 A^*)$ and the cross-orbit component is the residual.

Since the loss is constant along orbits, the gradient must be orthogonal to $T_{\mathrm{orbit}}$: for any $A$,
$$\frac{d}{dt}\Big|_{t=0} \mathcal{L}(e^{tA}W_1, W_2 e^{-tA}, W_3) = \langle \nabla_{W_1}\mathcal{L}, AW_1\rangle - \langle \nabla_{W_2}\mathcal{L}, W_2 A\rangle = 0.$$
So $A^* \approx 0$, the orbit component of the gradient is negligible, and stepping in the orbit direction should not change the loss. All loss reduction comes from the cross-orbit component.

We verify this by computing, at each training epoch, the loss decrease from: (1) the actual full gradient step, (2) a step using only the cross-orbit component, (3) a step using only the orbit component.


In [ ]:
# Retrain, but at each step decompose the JOINT gradient (dW1, dW2) into orbit
# and cross-orbit components in the joint parameter space.
#
# The orbit tangent space at (W1, W2) is {(A @ W1, -W2 @ A) : A in R^{n x n}}.
# We find A* that best approximates the joint gradient:
#   minimize ||A @ W1 - dW1||^2 + ||-W2 @ A - dW2||^2  over A
# via Kronecker product least-squares.

lr = 0.001
n_epochs = 3000
np.random.seed(42)

W1_t = 0.1 * np.random.randn(d_hidden, d_in)
W2_t = 0.1 * np.random.randn(d_hidden, d_hidden)
W3_t = 0.1 * np.random.randn(d_out, d_hidden)

rate_true = []       # actual loss change per epoch
rate_orbit = []      # loss change from orbit step
rate_cross = []      # loss change from cross-orbit step

prev_loss = None
for epoch in range(n_epochs):
    # Forward
    h1 = X @ W1_t.T
    h2 = h1 @ W2_t.T
    pred = h2 @ W3_t.T
    residual = pred - Y
    loss = (residual ** 2).mean()

    # Gradients
    dL_dpred = 2 * residual / n_samples
    dL_dW3 = dL_dpred.T @ h2
    dL_dh2 = dL_dpred @ W3_t
    dL_dW2 = dL_dh2.T @ h1
    dL_dh1 = dL_dh2 @ W2_t
    dL_dW1 = dL_dh1.T @ X

    # Solve for A*: Kronecker product least-squares
    n = d_hidden
    M1 = np.kron(W1_t.T, np.eye(n))
    M2 = -np.kron(np.eye(n), W2_t)
    M_stack = np.vstack([M1, M2])
    b_stack = np.concatenate([dL_dW1.ravel(), dL_dW2.ravel()])
    vec_A, _, _, _ = np.linalg.lstsq(M_stack, b_stack, rcond=None)
    A_opt = vec_A.reshape(n, n)

    # Orbit and cross-orbit components
    dW1_orbit = A_opt @ W1_t
    dW2_orbit = -W2_t @ A_opt
    dW1_cross = dL_dW1 - dW1_orbit
    dW2_cross = dL_dW2 - dW2_orbit

    # Loss after each type of step
    loss_after_orbit = compute_loss(
        W1_t - lr * dW1_orbit, W2_t - lr * dW2_orbit, W3_t, X, Y)
    loss_after_cross = compute_loss(
        W1_t - lr * dW1_cross, W2_t - lr * dW2_cross, W3_t - lr * dL_dW3, X, Y)

    # Rate = loss decrease from this step (positive = loss decreased)
    rate_orbit.append(loss - loss_after_orbit)
    rate_cross.append(loss - loss_after_cross)
    if prev_loss is not None:
        rate_true.append(prev_loss - loss)
    prev_loss = loss

    # Actual update
    W1_t -= lr * dL_dW1
    W2_t -= lr * dL_dW2
    W3_t -= lr * dL_dW3

rate_true = np.array(rate_true)
rate_orbit = np.array(rate_orbit[1:])   # align with rate_true
rate_cross = np.array(rate_cross[1:])

skip = 100
window = 50
smooth = lambda x: np.convolve(x, np.ones(window)/window, mode='valid')

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(smooth(np.abs(rate_true[skip:])), color='#2E86C1', linewidth=1.5,
            label='|Actual loss decrease|')
ax.semilogy(smooth(np.abs(rate_cross[skip:])), color='#27AE60', linewidth=1.5,
            linestyle='--', label='|Cross-orbit contribution|')
ax.semilogy(smooth(np.abs(rate_orbit[skip:])), color='#E74C3C', linewidth=1.5,
            linestyle='--', label='|Orbit contribution|')
ax.set_xlabel('Epoch')
ax.set_ylabel('|Loss decrease| per step (log scale)')
ax.set_title('Gradient decomposition: orbit contribution is orders of magnitude smaller')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Ex5_fig_gradient_decomposition.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print(f"Saved: Ex5_fig_gradient_decomposition.png")


## Summary

The loss landscape of a deep linear network has a rich geometric structure dictated by the $\mathrm{GL}(n)$ symmetry group:

- **Flat valleys** along orbits: reparametrizations that don't change the function. The loss is exactly constant (to machine precision) along these directions.
- **Steep walls** perpendicular to orbits: perturbations that change the actual linear map being computed.
- The orbit has dimension $n^2$ (the dimension of $\mathrm{GL}(n)$), so in a network with hidden dimension $n$, a massive $n^2$-dimensional subspace of parameter space is redundant.
- During training, gradient descent moves both along and across orbits. The along-orbit component is wasted — it explores equivalent solutions without improving the loss.

This is the reparametrization problem in its most concrete form. Activation functions and regularization shrink the symmetry group, reducing the dimension of these flat valleys and making optimization more efficient.
